# Validación del simulador para el estudio IBNR

Este cuaderno contiene únicamente las pruebas previas al experimento principal. Su propósito es verificar que el simulador construye triángulos de desarrollo coherentes con los supuestos definidos antes de ejecutar el diseño Monte Carlo completo.

La separación evita mezclar dos momentos metodológicos distintos: primero se revisa que el mecanismo de simulación funcione y después, en el notebook principal, se comparan los métodos de estimación del IBNR bajo escenarios contaminados.

## Alcance de este cuaderno

Este archivo no presenta los resultados finales del trabajo. Aqué se revisan tres elementos:

1. La configuración base del proceso generador de datos.
2. La validación interna del simulador frente a la esperanza teórica y la consistencia acumulada.
3. Una réplica ilustrativa para comprobar visualmente cómo se genera, observa y contamina un triángulo.

Los resultados finales, las métricas, los rankings y las conclusiones se presentan en `ibnr_chain_ladder_robusto.ipynb`.

## Ruta de este cuaderno dentro del proyecto

Este cuaderno cumple una funci?n preparatoria dentro del proyecto de grado:

1. Re?ne la misma configuraci?n base que se usar? en el experimento principal.
2. Verifica la coherencia interna del simulador frente a la teor?a planteada.
3. Muestra una r?plica ilustrativa para revisar c?mo se genera, se observa y se contamina un tri?ngulo.
4. Deja listo el paso al notebook principal, donde se ejecuta la comparaci?n Monte Carlo entre m?todos.

En otras palabras, aqu? se comprueba que el simulador funciona. En el cuaderno principal se responde la pregunta de investigaci?n.

In [ ]:
# Preparación de rutas del proyecto
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

OUTPUT_DIR = ROOT / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Librerías y funciones necesarias para validar el simulador
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from ibnr_project.config import build_default_config, build_default_scenarios
from ibnr_project.diagnostics import build_link_ratio_count_table
from ibnr_project.methods import estimate_ibnr_all_methods
from ibnr_project.simulation import observed_mask, simulate_single_triangle
from ibnr_project.validation import run_validation_suite

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titleweight"] = "bold"

## Configuración que será validada

La validación se realiza con la misma semilla, dimensión del triángulo, distribución base y matriz de escenarios usados en el experimento principal. Esto permite que la revisión previa y el análisis final trabajen bajo los mismos supuestos.

In [ ]:
# Definición de parámetros base para la prueba del simulador
SEED = 20260503
config = build_default_config(random_seed=SEED, distribution="gamma")
scenarios = build_default_scenarios()
mask = observed_mask(config.n_periods)

scenario_df = pd.DataFrame([scenario.__dict__ for scenario in scenarios])
scenario_df

## Estructura de los ratios de desarrollo

Antes de simular, se revisa cuántos ratios individuales quedan disponibles por periodo de desarrollo. Esta revisión es relevante porque los métodos robustos calculan medianas, medias truncadas o ponderaciones sobre esos conjuntos de ratios; cuando hay pocos datos, la estimación puede volverse más sensible.

In [ ]:
# Conteo de ratios individuales disponibles por periodo de desarrollo
ratio_count_df = build_link_ratio_count_table(mask)
ratio_count_df

In [ ]:
# Visualización del tamaño muestral disponible para estimar factores de desarrollo
plt.figure(figsize=(8, 4))
sns.barplot(data=ratio_count_df, x="development_period", y="n_individual_ratios", color="#35608d")
plt.title("Número de ratios individuales por periodo de desarrollo")
plt.xlabel("Periodo de desarrollo j")
plt.ylabel("Número de ratios C(i,j+1) / C(i,j)")
plt.tight_layout()
plt.show()

## Pruebas internas del simulador

Las siguientes pruebas revisan que el proceso generador de datos respete los supuestos usados en el diseño: coherencia con la esperanza teórica, consistencia entre montos incrementales y acumulados, y comportamiento razonable del IBNR verdadero construido por simulación.

In [ ]:
# Ejecución de pruebas de validación del simulador
validation_df = run_validation_suite(config)
validation_df

Las pruebas anteriores funcionan como control metodológico. Si el simulador no reprodujera de forma razonable la estructura esperada, las comparaciones entre métodos robustos y método clásico perderían sustento, porque el problema estaría en la generación de datos y no en el método de estimación.

## Inspección de una réplica simulada

Después de las pruebas agregadas, se observa una réplica individual. Esta revisión permite seguir el proceso completo a pequeña escala: se genera el triángulo completo, se oculta la parte futura, se introduce contaminación en la región observada y se conserva el IBNR verdadero para evaluar después los errores de estimación.

In [ ]:
# Generación de una réplica ilustrativa para revisar el flujo del simulador
rng_example = np.random.default_rng(SEED)
example = simulate_single_triangle(config, scenarios[4], rng_example)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(example.incremental_full, ax=axes[0], cmap="YlOrBr")
axes[0].set_title("Triángulo incremental completo")
sns.heatmap(example.observed_cumulative, ax=axes[1], cmap="Blues")
axes[1].set_title("Triángulo acumulado observado contaminado")
sns.heatmap(np.where(~example.observed_mask, example.incremental_full, np.nan), ax=axes[2], cmap="Reds")
axes[2].set_title("Región futura no observada")

for ax in axes:
    ax.set_xlabel("Periodo de desarrollo")
    ax.set_ylabel("Año de ocurrencia")

plt.tight_layout()
plt.show()

print(f"IBNR verdadero de la réplica: {example.true_ibnr:,.2f}")
print(f"Escenario revisado: {scenarios[4].name}")

La figura confirma la lógica triangular del ejercicio. El triángulo completo existe solo dentro de la simulación; el método de estimación observa únicamente la parte acumulada disponible. La región futura no observada permite calcular el IBNR verdadero, que después se usa como referencia para medir el error de cada método.

## Revisión de métodos sobre la réplica

Como éltima prueba, se aplican los métodos de estimación a la réplica ilustrativa. Esta salida no debe interpretarse como resultado final; solo verifica que todos los métodos reciben el mismo triángulo observado y producen una estimación comparable del IBNR.

In [ ]:
# Aplicación de los métodos de estimación sobre la réplica ilustrativa
estimate_preview = estimate_ibnr_all_methods(example.observed_cumulative, config)
estimate_preview

## Cierre de la validación

Con estas pruebas se deja documentado que el simulador genera datos coherentes con los supuestos del estudio y que el flujo computacional permite pasar de un triángulo simulado a estimaciones de IBNR comparables. Por esa razón, el notebook principal puede concentrarse en el experimento Monte Carlo, las métricas de desempeño y las conclusiones.